# 04 — Train 3D CNN

**Architecture:** Shallow 3D CNN with two output heads:
- **Regression head** — predicts trait anxiety, chronic stress, neuroticism (z-scored)
- **Network head** — predicts DMN, Salience, ECN activation levels (0–1)

**Why shallow?** LEMON has ~200 subjects. Deep networks overfit badly at this scale. We use:
- 3 convolutional blocks (32→64→128 filters)
- Dropout (0.4) and L2 regularisation
- 5-fold cross-validation
- Augmentation: random flips + small affine jitter

**Output:** `best_model.keras` saved to Google Drive — ready to serve via FastAPI.

In [ ]:
!pip install -q tensorflow numpy scikit-learn matplotlib

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, json, os
BASE_DIR  = '/content/drive/MyDrive/lemon'
MODEL_DIR = f'{BASE_DIR}/models'
os.makedirs(MODEL_DIR, exist_ok=True)

X     = np.load(f'{BASE_DIR}/X.npy')
y_reg = np.load(f'{BASE_DIR}/y_regression.npy')
y_net = np.load(f'{BASE_DIR}/y_network.npy')

print(f'X:     {X.shape}')
print(f'y_reg: {y_reg.shape}  (trait_anxiety, chronic_stress, neuroticism)')
print(f'y_net: {y_net.shape}  (DMN, Salience, ECN)')

In [ ]:
# ── Augmentation ──────────────────────────────────────────────────────────────
# Applied only during training. Brain scans are symmetric so L/R flips are valid.

def augment(x):
    """
    x: (2, 48, 48, 32) float32
    Returns augmented copy.
    """
    # Random left-right flip (axis 1 = x dimension)
    if np.random.random() < 0.5:
        x = x[:, ::-1, :, :]
    # Random anterior-posterior flip (axis 2 = y dimension)
    if np.random.random() < 0.5:
        x = x[:, :, ::-1, :]
    # Small intensity jitter (±2% of std)
    x = x + np.random.normal(0, 0.02, x.shape).astype(np.float32)
    return x

print('Augmentation function defined ✓')

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

def build_model(input_shape=(48, 48, 32, 2)):
    """
    3D CNN with two output heads.
    Input: (batch, 48, 48, 32, 2) — channels-last for TF
    """
    reg = regularizers.l2(1e-4)
    inp = keras.Input(shape=input_shape, name='bold_input')

    # ── Encoder ───────────────────────────────────────────────────────────────
    x = layers.Conv3D(32, 3, padding='same', activation='relu', kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(2)(x)               # → (24, 24, 16, 32)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv3D(64, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(2)(x)               # → (12, 12, 8, 64)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv3D(128, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)      # → (128,)  avoids overfitting vs Flatten
    x = layers.Dropout(0.4)(x)

    # ── Shared dense layer ────────────────────────────────────────────────────
    shared = layers.Dense(64, activation='relu', kernel_regularizer=reg, name='shared')(x)
    shared = layers.Dropout(0.3)(shared)

    # ── Regression head: trait anxiety, chronic stress, neuroticism ───────────
    reg_out = layers.Dense(32, activation='relu')(shared)
    reg_out = layers.Dense(3, name='wellbeing_output')(reg_out)  # linear, z-scored targets

    # ── Network head: DMN, Salience, ECN activation (0–1) ────────────────────
    net_out = layers.Dense(32, activation='relu')(shared)
    net_out = layers.Dense(3, activation='sigmoid', name='network_output')(net_out)

    model = keras.Model(inputs=inp, outputs=[reg_out, net_out])
    return model

model = build_model()
model.summary()

In [ ]:
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

# Reformat X from (N, 2, 48, 48, 32) → (N, 48, 48, 32, 2) for channels-last TF
X_tf = np.transpose(X, (0, 2, 3, 4, 1))

BATCH_SIZE = 8      # small batch fits T4 memory with 3D volumes
EPOCHS     = 60
N_FOLDS    = 5

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_histories = []
fold_val_losses = []

best_val_loss   = np.inf
best_model_path = f'{MODEL_DIR}/best_model.keras'

for fold, (train_idx, val_idx) in enumerate(kf.split(X_tf)):
    print(f'\n{"="*50}')
    print(f'FOLD {fold+1}/{N_FOLDS}  — train: {len(train_idx)}  val: {len(val_idx)}')
    print(f'{"="*50}')

    X_train, X_val   = X_tf[train_idx], X_tf[val_idx]
    yr_train, yr_val = y_reg[train_idx], y_reg[val_idx]
    yn_train, yn_val = y_net[train_idx], y_net[val_idx]

    # ── Apply augmentation to training set ────────────────────────────────────
    X_aug  = np.array([augment(x.transpose(3,0,1,2)).transpose(1,2,3,0) for x in X_train])
    X_train_all  = np.concatenate([X_train, X_aug], axis=0)
    yr_train_all = np.concatenate([yr_train, yr_train], axis=0)
    yn_train_all = np.concatenate([yn_train, yn_train], axis=0)

    # ── Build & compile fresh model per fold ─────────────────────────────────
    m = build_model()
    m.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={
            'wellbeing_output': 'mse',
            'network_output':   'mse',
        },
        loss_weights={'wellbeing_output': 1.0, 'network_output': 0.5},
        metrics={'wellbeing_output': 'mae', 'network_output': 'mae'},
    )

    callbacks = [
        keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss'),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-5, monitor='val_loss'),
    ]

    history = m.fit(
        X_train_all,
        {'wellbeing_output': yr_train_all, 'network_output': yn_train_all},
        validation_data=(
            X_val,
            {'wellbeing_output': yr_val, 'network_output': yn_val}
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    val_loss = min(history.history['val_loss'])
    fold_val_losses.append(val_loss)
    fold_histories.append(history.history)
    print(f'Fold {fold+1} best val_loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        m.save(best_model_path)
        print(f'  → New best model saved (val_loss={best_val_loss:.4f})')

print(f'\nCross-validation complete.')
print(f'Val losses per fold: {[f"{v:.4f}" for v in fold_val_losses]}')
print(f'Mean val loss: {np.mean(fold_val_losses):.4f} ± {np.std(fold_val_losses):.4f}')
print(f'Best model saved to: {best_model_path}')

In [ ]:
# ── Plot training curves for best fold ───────────────────────────────────────
best_fold = int(np.argmin(fold_val_losses))
h = fold_histories[best_fold]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h['loss'], label='train')
axes[0].plot(h['val_loss'], label='val')
axes[0].set_title(f'Total loss (fold {best_fold+1})')
axes[0].legend()

axes[1].plot(h['wellbeing_output_mae'], label='train')
axes[1].plot(h['val_wellbeing_output_mae'], label='val')
axes[1].set_title('Wellbeing MAE (z-scored)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/training_curves.png', dpi=100)
plt.show()
print('Next step: open 05_gradcam.ipynb')